# import datasets

In [22]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")

from src.data import load_data, split_by_date
from src.metrics import metrics

df = load_data(only_open=True)
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,3,2013-01-02,5530,668,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,1,4,2013-01-03,4327,578,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
2,1,5,2013-01-04,4486,619,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
3,1,6,2013-01-05,4997,635,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
4,1,1,2013-01-07,7176,785,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN


# baselines

In [23]:
cutoff = pd.Timestamp("2015-06-14")
train = df[df['Date'] < cutoff]
validation = df[df['Date'] >= cutoff]

In [24]:
# at store level
baseline_store = train.groupby("Store")["Sales"].mean()

# store + day of the week
baseline_store_dow = train.groupby(["Store", "DayOfWeek"])["Sales"].mean()

# only lasts weeks (8, 16)
baseline_8 = train[train['Date'] >= cutoff - pd.Timedelta(weeks=8)].groupby("Store")["Sales"].mean()
baseline_16 = train[train['Date'] >= cutoff - pd.Timedelta(weeks=16)].groupby("Store")["Sales"].mean()

# store + day of the week + promo
baseline_store_dow_promo = train.groupby(["Store", "DayOfWeek", "Promo"])["Sales"].mean()

# only lasts weeks (8, 16) + day of the week
baseline_8_dow = train[train['Date'] >= cutoff - pd.Timedelta(weeks=8)].groupby(["Store", "DayOfWeek"])["Sales"].mean()
baseline_16_dow = train[train['Date'] >= cutoff - pd.Timedelta(weeks=16)].groupby(["Store", "DayOfWeek"])["Sales"].mean()

# only lasts weeks (8, 16) + day of the week + promo
baseline_8_dow_promo = train[train['Date'] >= cutoff - pd.Timedelta(weeks=8)].groupby(["Store", "DayOfWeek", "Promo"])["Sales"].mean()
baseline_16_dow_promo = train[train['Date'] >= cutoff - pd.Timedelta(weeks=16)].groupby(["Store", "DayOfWeek", "Promo"])["Sales"].mean()

# seasonal naive
baseline_seasonal = train.sort_values("Date").groupby(["Store", "DayOfWeek"])["Sales"].last()

valid = validation.merge(baseline_store_dow, on=["Store", "DayOfWeek"], suffixes=("", "_baseline_store_dow"), how="left") \
    .merge(baseline_store, on="Store", suffixes=("", "_baseline_store"), how="left") \
    .merge(baseline_8, on="Store", suffixes=("", "_baseline_8"), how="left") \
    .merge(baseline_16, on="Store", suffixes=("", "_baseline_16"), how="left") \
    .merge(baseline_store_dow_promo, on=["Store", "DayOfWeek", "Promo"], suffixes=("", "_baseline_store_dow_promo"), how="left") \
    .merge(baseline_8_dow, on=["Store", "DayOfWeek"], suffixes=("", "_baseline_8_dow"), how="left") \
    .merge(baseline_16_dow, on=["Store", "DayOfWeek"], suffixes=("", "_baseline_16_dow"), how="left") \
    .merge(baseline_8_dow_promo, on=["Store", "DayOfWeek", "Promo"], suffixes=("", "_baseline_8_dow_promo"), how="left") \
    .merge(baseline_16_dow_promo, on=["Store", "DayOfWeek", "Promo"], suffixes=("", "_baseline_16_dow_promo"), how="left") \
    .merge(baseline_seasonal, on=["Store", "DayOfWeek"], suffixes=("", "_baseline_seasonal"), how="left")

# Metrics

In [25]:

res = pd.concat({
    "store":     metrics(valid["Sales"], valid["Sales_baseline_store"]),
    "store_dow": metrics(valid["Sales"], valid["Sales_baseline_store_dow"]),
    "store_dow_promo": metrics(valid["Sales"], valid["Sales_baseline_store_dow_promo"]),
    "store_8":   metrics(valid["Sales"], valid["Sales_baseline_8"]),
    "store_16":  metrics(valid["Sales"], valid["Sales_baseline_16"]),
    "store_8_dow":   metrics(valid["Sales"], valid["Sales_baseline_8_dow"]),
    "store_16_dow":  metrics(valid["Sales"], valid["Sales_baseline_16_dow"]),
    "store_8_dow_promo":   metrics(valid["Sales"], valid["Sales_baseline_8_dow_promo"]),
    "store_16_dow_promo":  metrics(valid["Sales"], valid["Sales_baseline_16_dow_promo"]),
    "store_seasonal":  metrics(valid["Sales"], valid["Sales_baseline_seasonal"])
})
res

,,RMSPE,RMSE,MAE,WAPE
store,0,0.318240,1960.229183,1467.905480,0.205435
store_dow,0,0.228268,1665.984830,1259.299892,0.176240
store_dow_promo,0,0.142878,1090.145361,737.667038,0.103237
store_8,0,0.321193,1893.109201,1442.070472,0.201819
store_16,0,0.317959,1893.981686,1436.515420,0.201042
store_8_dow,0,0.232312,1614.044175,1241.821823,0.173794
store_16_dow,0,0.228720,1602.833207,1235.228371,0.172871
store_8_dow_promo,0,0.152774,1085.566599,768.373645,0.107535
store_16_dow_promo,0,0.143793,1030.912943,718.650334,0.100576
store_seasonal,0,0.230477,2083.536277,1466.104241,0.205183


The best baseline is calculated by store, day of the week and promotion with RMSPE of 0.143 and WAPE of 0.103. \
Eastern and Christmas do not belong to the validation set so including them is useless for such predictions. \
With more data and predictions across the whole year it would make sense to feature engineer thoses periods.

# import functions

In [26]:
%load_ext autoreload
%autoreload 2

# import data
df = load_data(only_open=True)

# split data
train, validation = split_by_date(df, '2015-06-14')

# baseline store + day of the week + promo
baseline_store_dow_promo = train.groupby(["Store", "DayOfWeek", "Promo"])["Sales"].mean()
valid = validation.merge(baseline_store_dow_promo, on=["Store", "DayOfWeek", "Promo"], suffixes=("", "_baseline_store_dow_promo"), how="left")

# metrics
res = pd.concat({
    "store_dow_promo": metrics(valid["Sales"], valid["Sales_baseline_store_dow_promo"]),
})
res




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,,RMSPE,RMSE,MAE,WAPE
store_dow_promo,0,0.142878,1090.145361,737.667038,0.103237


In [27]:
valid[["Sales_baseline_store_dow_promo"]].isna().sum()

Sales_baseline_store_dow_promo    0
dtype: int64